# Extra · Week 1 — Mean reversion: the opposite bet

**Optional.** For anyone who's finished the core week-1 notebooks and wants
another real strategy to build.

SMA crossover (what you built this week) is **trend-following**: it bets a stock
going up will keep going up. This notebook builds the opposite philosophy —
**mean reversion**: the bet that a stock pushed too far, too fast, tends to snap
back toward normal. Same market, same data, completely different theory of how
prices behave.

One function you build: `rsi_mean_reversion_weights` — and it reuses the RSI you
already wrote in week 1, this time as a real trading signal instead of just a
chart overlay.

## 1. The idea

RSI (you built this in NB2) measures how hard a stock has been pushed lately, on
a 0–100 scale. Below ~30 is called **oversold** — sold off hard, possibly overdone.
Above ~70 is **overbought**.

Trend-following says: "it's falling, stay away." Mean reversion says the opposite:
"it's been pushed down hard — that's often an overreaction, and overreactions tend
to correct." Both are real, widely used trading philosophies. Neither is obviously
right; which one works better depends on the stock, the market regime, and the
timeframe — which is exactly why it's worth building both and comparing honestly.

In [ ]:
import sys, os
while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from tradinglab.data_feed import DataFeed
feed = DataFeed.from_dir('data/egx', symbols=['COMI', 'HRHO', 'TMGH', 'SWDY', 'FWRY', 'ABUK'])

from tradinglab.observation import build_observation
obs_today = build_observation(feed, feed.n_days - 2, lookback=30)
rsi_today = obs_today[:, -1, 3]   # feature 3 = RSI, already computed — your week-1 function
for sym, r in zip(feed.symbols, rsi_today):
    tag = 'OVERSOLD' if r < 30 else 'overbought' if r > 70 else ''
    print(f'{sym:6s} RSI = {r:5.1f}  {tag}')

## 2. Function — `rsi_mean_reversion_weights`

**Rule:** find stocks with RSI below the `oversold` threshold (default 30). Among
those, hold the `top_k` MOST oversold (lowest RSI), equal-weight. If nothing is
oversold today, hold everything equally rather than force a bet that isn't there.

**In:** `observation` (feature 3 is RSI), `oversold` threshold, `top_k`.
**Out:** weights `(n_assets,)`, non-negative, summing to 1.
**Hint:** `np.where(rsi_today < oversold)[0]` gives you the candidates;
`np.argsort` ranks them from most to least oversold.
**Done when:** the check passes.

In [ ]:
def rsi_mean_reversion_weights(observation, oversold=30.0, top_k=2):
    n_assets = observation.shape[0]
    rsi_today = observation[:, -1, 3]

    # ---8<--- solution
    oversold_mask = rsi_today < oversold
    candidates = np.where(oversold_mask)[0]

    if len(candidates) == 0:
        return np.ones(n_assets) / n_assets

    ranked = candidates[np.argsort(rsi_today[candidates])]
    chosen = ranked[:top_k]

    weights = np.zeros(n_assets)
    weights[chosen] = 1.0 / len(chosen)
    return weights
    # ---8<--- end

# check: a made-up situation where you KNOW who should be picked
test_obs = np.zeros((5, 30, 5))
test_obs[:, -1, 3] = [80, 20, 65, 10, 50]     # RSI per stock
w = rsi_mean_reversion_weights(test_obs, oversold=30, top_k=2)
assert abs(w.sum()-1) < 1e-9 and w[3] > 0 and w[1] > 0 and w[0] == 0, 'not right yet'
print('rsi_mean_reversion_weights correct ✓  — picked the two most oversold')

## 3. The showdown — same universe, same money, opposite theories

Run both strategies through the identical backtester, on the identical 5 stocks,
against the identical real EGX30 benchmark. This is a fair fight.

In [ ]:
from tradinglab.simulator import PortfolioSimulator
from tradinglab.backtester import run_backtest
from tradinglab.strategies.sma import sma_crossover_weights

sim = PortfolioSimulator(feed, benchmark='egx30')

res_trend = run_backtest(sim, lambda o: sma_crossover_weights(o, 9, 20), lookback=30)
res_revert = run_backtest(sim, lambda o: rsi_mean_reversion_weights(o, oversold=30, top_k=2), lookback=30)

START = 1000.0
plt.figure(figsize=(11, 5))
plt.plot(res_trend['dates'], res_trend['portfolio'] * START, label='trend-following (SMA)')
plt.plot(res_revert['dates'], res_revert['portfolio'] * START, label='mean-reversion (RSI)')
plt.plot(res_trend['dates'], res_trend['benchmark'] * START, label='EGX30 benchmark', linestyle='--')
plt.legend(); plt.grid(alpha=0.3); plt.title('Two philosophies, same money, same market')
plt.ylabel('EGP'); plt.gcf().autofmt_xdate()
plt.show()

In [ ]:
from tradinglab import metrics

for name, res in [('trend-following', res_trend), ('mean-reversion', res_revert)]:
    r = res['portfolio_returns']
    final = res['portfolio'][-1] * START
    print(f"{name:16s}  final: {final:7,.0f} EGP  |  sharpe: {metrics.sharpe(r):5.2f}  |  max DD: {metrics.max_drawdown(r):.1%}")

bench_final = res_trend['benchmark'][-1] * START
print(f"{'EGX30 benchmark':16s}  final: {bench_final:7,.0f} EGP")

## 4. Reflect

Neither philosophy is "correct" in general — that's the honest answer, and it's
worth sitting with rather than looking for a winner. What's worth noticing instead:

- **Did they behave differently, not just score differently?** Look at the shapes
  of the two curves, not just the endpoints. Trend-following tends to do well in
  strong, sustained moves and badly in choppy markets. Mean reversion tends to do
  the opposite.
- **Try changing `oversold` and `top_k`.** A stricter threshold (say 20 instead of
  30) trades less often, on stronger signals — does that help or hurt?
- **The real test — same as everywhere else this week — is whether either one
  beats simply buying and holding these 5 stocks.** You already know the answer
  for trend-following from NB4. Check it for mean-reversion the same way.

**Graduate it** into `src/tradinglab/strategies/mean_reversion.py` if you want it
on your dashboard leaderboard alongside SMA — same pattern as everything else this
week.